# 01 — EIA Data Collection

Этот ноутбук собирает базовые данные из **U.S. Energy Information Administration (EIA) Open Data API v2** для проекта по прогнозированию изменения нефтяных запасов и реакции WTI calendar spread.

На этом этапе мы:

1. проверяем Python и виртуальное окружение;
2. находим корень проекта;
3. подключаем `EIAClient`;
4. проверяем соединение с EIA API;
5. загружаем недельные нефтяные данные;
6. загружаем исторические WTI futures F1–F4;
7. считаем изменения запасов;
8. считаем calendar spreads;
9. проверяем качество данных;
10. сохраняем результаты в `data/raw/eia/`.

> API-ключ **не хранится в ноутбуке**. Он должен находиться в `.env` в корне проекта:
>
> `EIA_API_KEY=...`


## 1. Проверка окружения

Ноутбук должен работать из виртуального окружения проекта `.venv` на Python 3.14.

Если здесь отображается другой Python, в VS Code нужно выбрать правильный kernel.


In [ ]:
import sys
from pathlib import Path

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)


Ожидаемый путь к интерпретатору должен содержать примерно:

```text
oil-news-graph/.venv/bin/python
```

Если используется системный Python, сначала выбери kernel проекта.


## 2. Импорт библиотек и поиск корня проекта

Ноутбук может лежать не прямо в папке `notebooks/`, поэтому корень проекта ищем автоматически.

Корнем считаем ближайшую родительскую директорию, где находится папка `src`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start: Path | None = None) -> Path:
    """Ищет ближайшую родительскую папку, содержащую src/."""

    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "Не удалось найти корень проекта. "
        "Проверь, что в проекте есть папка src/."
    )


ROOT = find_project_root()

# Добавляем корень проекта в sys.path,
# чтобы Python видел src/data/eia_client.py
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw" / "eia"
PROCESSED_DIR = ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw data directory:", RAW_DIR)


## 3. Подключение EIA-клиента

Используем готовый файл:

```text
src/data/eia_client.py
```

В нём уже находятся:

- `EIAClient`;
- `EIA_WEEKLY_SERIES`;
- `EIA_WTI_FUTURES_SERIES`.

Если ты только что менял `eia_client.py`, лучше перезапустить kernel перед запуском ноутбука.


In [ ]:
from src.data.eia_client import (
    EIAClient,
    EIA_WEEKLY_SERIES,
    EIA_WTI_FUTURES_SERIES,
)

client = EIAClient()

print("EIAClient initialized successfully.")


## 4. Проверка соединения

Перед массовой загрузкой всегда проверяем:

- читается ли `.env`;
- работает ли API-ключ;
- отвечает ли EIA API;
- доступен ли тестовый petroleum route.


In [ ]:
client.test_connection()


Если результат:

```text
True
```

— соединение работает.

Если возникает `401` или `403`, проблема обычно в API-ключе.

Если возникает `404`, проблема обычно в route или series ID.


## 5. Период исследования

Основной период проекта начинается с **19 февраля 2015 года**.

Эта дата удобна тем, что совпадает с началом GDELT 2.0, который позже будет использоваться для новостного графа.

Конец периода пока задаём текущей датой исследования.


In [ ]:
START_DATE = "2015-02-19"
END_DATE = "2026-11-03"

print("Start:", START_DATE)
print("End:", END_DATE)


## 6. Посмотрим, какие weekly series сейчас настроены

Не создаём `WEEKLY_SERIES` вручную в ноутбуке.

Используем словарь из `eia_client.py`, чтобы не было двух разных версий конфигурации.


In [ ]:
EIA_WEEKLY_SERIES


## 7. Сначала тестируем один ряд

Перед `fetch_many()` полезно проверить основной ряд отдельно.

Основной target-source:

**U.S. commercial crude oil stocks excluding SPR**

Series:

```text
WCESTUS1
```


In [ ]:
test_crude_stocks = client.fetch_series(
    route="petroleum/stoc/wstk",
    series="WCESTUS1",
    frequency="weekly",
    start="2025-01-01",
    end="2025-03-31",
)

test_crude_stocks.head()


In [ ]:
print("Shape:", test_crude_stocks.shape)
print(
    "Period:",
    test_crude_stocks["period"].min(),
    "->",
    test_crude_stocks["period"].max(),
)


Если здесь появилась таблица, можно переходить к полной weekly-загрузке.


# 8. Загрузка weekly EIA data

Загружаем все серии, которые сейчас находятся в `EIA_WEEKLY_SERIES`.

На первом этапе это:

- commercial crude stocks;
- Cushing stocks;
- PADD 1–5 stocks;
- U.S. crude production.

Позже словарь можно расширять imports, exports, refinery inputs, refinery utilization, gasoline, distillates и другими фундаментальными рядами.


In [ ]:
weekly = client.fetch_many(
    series=EIA_WEEKLY_SERIES,
    start=START_DATE,
    end=END_DATE,
)

weekly.head()


In [ ]:
print("Shape:", weekly.shape)

print(
    "Period:",
    weekly["period"].min(),
    "->",
    weekly["period"].max(),
)


## 9. Проверка weekly dataset

Смотрим:

- типы данных;
- пропуски;
- дубликаты дат;
- последние наблюдения.


In [ ]:
weekly.info()


In [ ]:
weekly.tail(10)


In [ ]:
weekly.isna().sum().sort_values(ascending=False)


In [ ]:
duplicate_dates = weekly["period"].duplicated().sum()

print("Duplicated dates:", duplicate_dates)

assert duplicate_dates == 0, "В weekly dataset есть дубликаты дат."


## 10. Проверка недельных интервалов

Большинство соседних observation dates должны отличаться примерно на 7 дней.

Отдельные отклонения могут появляться из-за особенностей публикации или структуры ряда, поэтому их не удаляем автоматически.


In [ ]:
weekly_intervals = weekly["period"].diff().dt.days

weekly_intervals.value_counts().sort_index()


# 11. Подготовка запасов

EIA stock series обычно выражены в **thousand barrels**.

Для дальнейшей работы удобнее также иметь значения в **million barrels (MMbbl)**.

Перевод:

```text
1 MMbbl = 1000 thousand barrels
```


In [ ]:
weekly_features = weekly.copy()

stock_columns = [
    col
    for col in [
        "crude_stocks",
        "cushing_stocks",
        "padd1_stocks",
        "padd2_stocks",
        "padd3_stocks",
        "padd4_stocks",
        "padd5_stocks",
    ]
    if col in weekly_features.columns
]

for col in stock_columns:
    weekly_features[f"{col}_mmbbl"] = (
        weekly_features[col] / 1000
    )

stock_columns


## 12. Изменение коммерческих запасов

Наш будущий target связан не с уровнем запасов, а с **еженедельным изменением запасов**.

Сначала считаем фактическое изменение:

```text
Actual Δ Stocks_t = Stocks_t - Stocks_(t-1)
```

Позже, когда добавим analyst consensus:

```text
Inventory Surprise_t =
Actual Δ Stocks_t - Consensus_t
```


In [ ]:
if "crude_stocks_mmbbl" in weekly_features.columns:
    weekly_features["crude_stocks_change_mmbbl"] = (
        weekly_features["crude_stocks_mmbbl"].diff()
    )

if "cushing_stocks_mmbbl" in weekly_features.columns:
    weekly_features["cushing_change_mmbbl"] = (
        weekly_features["cushing_stocks_mmbbl"].diff()
    )

for padd in range(1, 6):
    col = f"padd{padd}_stocks_mmbbl"

    if col in weekly_features.columns:
        weekly_features[f"padd{padd}_change_mmbbl"] = (
            weekly_features[col].diff()
        )


In [ ]:
cols_to_show = [
    col
    for col in [
        "period",
        "crude_stocks_mmbbl",
        "crude_stocks_change_mmbbl",
        "cushing_stocks_mmbbl",
        "cushing_change_mmbbl",
        "crude_production",
    ]
    if col in weekly_features.columns
]

weekly_features[cols_to_show].tail(10)


## 13. Визуальная проверка commercial crude stocks


In [ ]:
if "crude_stocks_mmbbl" in weekly_features.columns:
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(
        weekly_features["period"],
        weekly_features["crude_stocks_mmbbl"],
    )

    ax.set_title(
        "U.S. Commercial Crude Oil Stocks excluding SPR"
    )
    ax.set_xlabel("Week ending")
    ax.set_ylabel("Million barrels")

    plt.show()


# 14. Загрузка исторических WTI futures

EIA публиковала ежедневные цены первых четырёх последовательных WTI futures contracts:

- `RCLC1` → F1;
- `RCLC2` → F2;
- `RCLC3` → F3;
- `RCLC4` → F4.

Для стратегии нас интересует не столько абсолютная цена WTI, сколько **calendar spread**:

```text
Spread = F_near - F_far
```

Например:

```text
F1 - F3
```

Интерпретация:

- spread > 0 → backwardation;
- spread < 0 → contango.


In [ ]:
EIA_WTI_FUTURES_SERIES


## 15. Сначала проверяем один futures series


In [ ]:
test_f1 = client.fetch_series(
    route="petroleum/pri/fut",
    series="RCLC1",
    frequency="daily",
    start="2024-01-01",
    end="2024-04-30",
)

test_f1.tail()


In [ ]:
print("F1 observations:", len(test_f1))

if not test_f1.empty:
    print(
        "F1 period:",
        test_f1["period"].min(),
        "->",
        test_f1["period"].max(),
    )


Исторические futures series EIA заканчиваются раньше текущего периода проекта.

Это нормально: данные EIA F1–F4 будем использовать для бесплатного исторического EDA, а более поздний период позже дополним отдельным источником.


## 16. Загружаем F1–F4


In [ ]:
futures = client.fetch_many(
    series=EIA_WTI_FUTURES_SERIES,
    start=START_DATE,
    end=END_DATE,
)

futures.head()


In [ ]:
print("Shape:", futures.shape)

print(
    "Period:",
    futures["period"].min(),
    "->",
    futures["period"].max(),
)


In [ ]:
futures.tail(10)


# 17. Calendar spreads

Считаем:

```text
F1-F2
F1-F3
F1-F4
```

Для основной гипотезы сначала особенно интересен `F1-F3`.

Мы используем соглашение:

```text
spread = near contract - far contract
```


In [ ]:
futures_features = futures.copy()

required_pairs = [
    ("wti_f1", "wti_f2", "spread_1_2"),
    ("wti_f1", "wti_f3", "spread_1_3"),
    ("wti_f1", "wti_f4", "spread_1_4"),
]

for near_col, far_col, spread_col in required_pairs:
    if (
        near_col in futures_features.columns
        and far_col in futures_features.columns
    ):
        futures_features[spread_col] = (
            futures_features[near_col]
            - futures_features[far_col]
        )

futures_features.tail()


## 18. Изменение spread

Позже будем сопоставлять surprise в запасах с изменением кривой WTI.

Поэтому считаем:

```text
ΔSpread_t = Spread_t - Spread_(t-1)
```


In [ ]:
spread_columns = [
    col
    for col in [
        "spread_1_2",
        "spread_1_3",
        "spread_1_4",
    ]
    if col in futures_features.columns
]

for col in spread_columns:
    futures_features[f"delta_{col}"] = (
        futures_features[col].diff()
    )

futures_features.tail()


## 19. Визуальная проверка F1-F3 spread


In [ ]:
if "spread_1_3" in futures_features.columns:
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(
        futures_features["period"],
        futures_features["spread_1_3"],
    )

    ax.axhline(0, linewidth=1)

    ax.set_title(
        "WTI Calendar Spread: F1 - F3"
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("USD / barrel")

    plt.show()


# 20. Функция для быстрого контроля качества

Эта таблица показывает:

- dtype;
- количество пропусков;
- процент пропусков;
- число уникальных значений.


In [ ]:
def data_summary(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing": df.isna().sum(),
            "missing_pct": (
                df.isna().mean() * 100
            ).round(2),
            "n_unique": df.nunique(dropna=True),
        }
    )


data_summary(weekly_features)


In [ ]:
data_summary(futures_features)


# 21. Сохранение данных

Сохраняем как:

- CSV — удобно быстро открыть глазами;
- Parquet — основной формат для дальнейшего Python pipeline.

Большие файлы из `data/raw/` обычно **не пушим в Git**.


In [ ]:
weekly_csv = RAW_DIR / "eia_weekly_2015_2026.csv"
weekly_parquet = RAW_DIR / "eia_weekly_2015_2026.parquet"

weekly_features.to_csv(
    weekly_csv,
    index=False,
)

weekly_features.to_parquet(
    weekly_parquet,
    index=False,
)

print("Saved:")
print(weekly_csv)
print(weekly_parquet)


In [ ]:
futures_csv = RAW_DIR / "eia_wti_futures_2015_2024.csv"
futures_parquet = RAW_DIR / "eia_wti_futures_2015_2024.parquet"

futures_features.to_csv(
    futures_csv,
    index=False,
)

futures_features.to_parquet(
    futures_parquet,
    index=False,
)

print("Saved:")
print(futures_csv)
print(futures_parquet)


# 22. Итог текущего этапа

После успешного выполнения ноутбука у нас есть два основных блока данных.

### Weekly fundamentals

Примерно:

```text
data/raw/eia/eia_weekly_2015_2026.parquet
```

Содержит:

- commercial crude stocks;
- Cushing stocks;
- PADD stocks;
- production;
- производные weekly changes.

### WTI futures

```text
data/raw/eia/eia_wti_futures_2015_2024.parquet
```

Содержит:

- F1;
- F2;
- F3;
- F4;
- F1-F2;
- F1-F3;
- F1-F4;
- изменения spreads.

---

## Что пока НЕ делаем

В этом ноутбуке пока нет:

1. analyst consensus;
2. точного EIA `release_datetime`;
3. API Weekly Statistical Bulletin;
4. новостей;
5. графа;
6. merge fundamentals + consensus + market reaction.

Это будут отдельные этапы, чтобы не смешивать сбор разных источников в одном ноутбуке.

Следующий логичный шаг после проверки этого ноутбука — расширить `EIA_WEEKLY_SERIES` корректными native API v2 routes для:

- imports;
- exports;
- refinery inputs;
- refinery utilization;
- gasoline stocks;
- distillate stocks;
- SPR;
- products supplied.
